# 🏗️ Notebook 1: Gmail — Requirements & Architecture

> Part of the beginner-friendly **bad → best** progression. We start with a naive
> single-server inbox and evolve it into a globally-scalable email service.

## 🛠️ Setup

```bash
cd 06-system-designs/gmail
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 1. What are we designing?

A web-based email service (a simplified Gmail) that can:

- **Send** email to any domain on the Internet using **SMTP**.
- **Receive** email from any domain (be an **MX** host) and store it safely.
- Let users **read, search, label, and thread** their mail via a web UI / IMAP.
- Handle **attachments**, **spam**, and keep mail private and durable.

### Functional requirements
- Send / receive email (SMTP).
- Full-text **search** across the mailbox.
- **Labels / folders**, **threading** (conversations), **drafts**, **attachments**.
- Mobile + web client support (via **IMAP** or proprietary REST API).

### Non-functional requirements
- **Durability** — never lose a user's email (replicated, backed up).
- **Availability** — inbox should load in < 500 ms p99.
- **Scale** — ~1 B users, ~15 GB average mailbox → ~15 EB total.
- **Privacy** — strong tenancy isolation; spam + virus filtering.

### Protocols cheat-sheet

| Protocol | Port | Use |
|---|---|---|
| SMTP (server ↔ server) | 25 | Incoming mail delivery between mail servers |
| SMTP Submission | 587 (STARTTLS) / 465 (TLS) | Authenticated client → our server |
| IMAP | 993 (TLS) | Client reads server-side mailbox |
| POP3 | 995 (TLS) | Legacy: client downloads + deletes |

### Authenticity protocols (anti-spoofing)
- **SPF** (Sender Policy Framework) — DNS record lists IPs allowed to send for a domain.
- **DKIM** (DomainKeys Identified Mail) — cryptographic signature over headers + body.
- **DMARC** — policy tying SPF + DKIM to how receivers should react on failure.

## 2. Back-of-the-envelope capacity estimation

Let's actually **compute** the numbers — this is the kind of math that interviewers
love, and it tells us which parts of the system need the most engineering.

In [1]:
# Capacity estimation — plug in your own numbers at the top and rerun.

users           = 1_000_000_000          # 1B active users
avg_mailbox_gb  = 15                     # average size per inbox
msgs_per_day    = 40                     # received per user
avg_msg_kb      = 75                     # typical email size (no attachment)
attach_ratio    = 0.20                   # 20% of emails have an attachment
avg_attach_mb   = 0.5                    # average attachment size
peak_ratio      = 3                      # peak QPS / average QPS

# ----- storage -----
total_storage_eb = users * avg_mailbox_gb / 1e9          # GB → EB
daily_new_gb     = users * msgs_per_day * avg_msg_kb / 1e6
daily_attach_tb  = users * msgs_per_day * attach_ratio * avg_attach_mb / 1e6

# ----- traffic -----
daily_msgs       = users * msgs_per_day
writes_per_sec   = daily_msgs / 86_400
reads_per_sec    = writes_per_sec * 10                   # users read ~10x more than they receive
peak_reads       = reads_per_sec * peak_ratio

print(f"total mailbox storage : {total_storage_eb:7.1f} EB")
print(f"daily new mail        : {daily_new_gb:7.0f} GB  (+{daily_attach_tb:.0f} TB attachments)")
print(f"avg write QPS         : {writes_per_sec:7.0f} msgs/sec")
print(f"avg read  QPS         : {reads_per_sec:7.0f} ops/sec")
print(f"peak read QPS         : {peak_reads:7.0f} ops/sec")

total mailbox storage :    15.0 EB
daily new mail        : 3000000 GB  (+4000 TB attachments)
avg write QPS         :  462963 msgs/sec
avg read  QPS         : 4629630 ops/sec
peak read QPS         : 13888889 ops/sec


**Takeaways from the numbers**

- We can't put all mail in one DB: storage is **exabytes**, reads are **hundreds of thousands of QPS**.
- Most of the volume is attachments → send them to **object storage** (S3/GCS), not the DB.
- Read-heavy → aggressive **caching** of inbox headers.

## 3. Architecture — **bad → best** progression

We'll start as simply as possible and only add complexity when we *need* it.
This is a deliberate pedagogy: every box on the final diagram should have a reason.

### v1 — naive single-server inbox (BAD)

```
browser ──▶ monolith (SMTP + web + SQL) ──▶ single Postgres
```

Problems:
1. Incoming SMTP goes down → we **drop mail** (against RFC; senders will keep retrying but
   we're violating durability expectations).
2. Single DB → one noisy user can saturate the whole cluster.
3. Attachments inflate DB size 100× → backups become impossible.
4. No spam filtering → inbox drowns in junk.
5. No search index → every query is a SQL `LIKE '%...%'` full-table scan.

### v2 — split reads/writes + object store (BETTER)

```
browser ──▶ Web API ──┐
                       ├──▶ Metadata DB (rows per message)
SMTP in ──▶ Queue ─────┤
                       └──▶ Object store (bodies + attachments)
```

We decouple **inbound SMTP** from application writes via a **durable queue** (Kafka /
SQS). If the metadata DB is slow, mail doesn't bounce — it waits.

### v3 — sharded, indexed, tiered (BEST)

```
                  ┌────────────────┐
  MX DNS ────────▶│ SMTP inbound   │──┐
                  │ (accept + queue)│  │
                  └────────────────┘  │
                                      ▼
                               ┌──────────────┐
                               │ Spam / virus │──► quarantine
                               │  classifier  │
                               └──────┬───────┘
                                      │ clean
                                      ▼
  ┌─────────┐   ┌───────────────┐   ┌───────────────┐   ┌───────────────┐
  │ Web/IMAP│◀──│ Mailbox API   │──▶│ Metadata DB   │   │ Object Store  │
  │  (read) │   │  (messages,   │   │ sharded by    │   │ (bodies +     │
  │         │   │   threads)    │   │ user_id       │   │  attachments) │
  └─────────┘   └──────┬────────┘   └───────┬───────┘   └───────▲───────┘
                       │                    │                   │
                       │ new-msg event      │                   │ read/write refs
                       ▼                    ▼                   │
                 ┌───────────┐        ┌───────────┐             │
                 │ Search    │        │  Cache    │             │
                 │ indexer   │──▶ ES  │ (Redis)   │             │
                 └───────────┘        └───────────┘             │
                                                                │
  Outbound: client ──▶ SMTP submission (587) ──▶ relay ──▶ recipient MX
```

Key decisions encoded in the picture:
- **Partition key = `user_id`** — keeps all of one user's mail co-located.
- **Metadata vs body split** — hot fields in DB, fat fields in object storage.
- **Search is async** — we don't block inbound mail on indexing.
- **Spam runs before storage** — spam still goes to the spam folder (must be queryable),
  but the classifier decides the label at ingest time.

## 4. Why these choices?

| Concern | Naive approach | Our approach | Why |
|---|---|---|---|
| Durability | sync write to DB | accept → queue → persist | SMTP senders time out; queue absorbs bursts |
| Search | `LIKE '%x%'` | per-user inverted index | SQL scan is O(mailbox); index is O(log n) |
| Storage | all in DB | hot/warm/cold tiers | 99% of reads hit < 5% of data |
| Privacy | shared index | per-user shard | cross-tenant bugs can't leak data |
| Attachments | in message row | object store + hash-dedup | same attachment forwarded 1000× stored once |

In the next notebook we'll put these decisions into concrete data models and APIs.